# Chunking Verification

**Purpose:** Verify chunk quality on the real prospectus — size distribution, content coherence, and metadata correctness.

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from app.ingestion.pipeline import ingest_document
from app.ingestion.chunker import summarize_chunks

PDF_PATH = project_root / "data" / "raw_documents" / "Prospectus - FALL 2026 (29-07-2026).pdf"
chunks = ingest_document(PDF_PATH)
summary = summarize_chunks(chunks)

print("=== Chunk Summary ===")
for k, v in summary.items():
    print(f"  {k}: {v}")

## Size Distribution

In [ ]:
sizes = [c.char_count for c in chunks]

buckets = {"<200": 0, "200-400": 0, "400-600": 0, "600-800": 0, ">800": 0}
for s in sizes:
    if s < 200: buckets["<200"] += 1
    elif s < 400: buckets["200-400"] += 1
    elif s < 600: buckets["400-600"] += 1
    elif s <= 800: buckets["600-800"] += 1
    else: buckets[">800"] += 1

print("Size distribution:")
for label, count in buckets.items():
    bar = "#" * (count // 2)
    print(f"  {label:>8}: {count:>4}  {bar}")

## Sample Chunks — first, middle, last

In [ ]:
def show_chunk(chunk, max_chars=600):
    text = chunk.text[:max_chars]
    suffix = "... [truncated]" if len(chunk.text) > max_chars else ""
    print(f"{'=' * 70}")
    print(f"CHUNK {chunk.chunk_index}  |  {chunk.char_count} chars  |  pages: {chunk.page_numbers}  |  source: {chunk.source_document}")
    print(f"{'=' * 70}")
    print(text + suffix)
    print()

for idx in [0, len(chunks) // 2, len(chunks) - 1]:
    show_chunk(chunks[idx])

## Header/Footer Leak Check
Verify no chunks still contain the recurring header/footer phrase.

In [ ]:
leaked = [c.chunk_index for c in chunks if "UNIVERSITY OF EDUCATION, LAHORE" in c.text]
print(f"Chunks with leftover header/footer: {len(leaked)}")
if leaked:
    print(f"  Indices: {leaked[:20]}")